<a href="https://colab.research.google.com/github/AristidesGuilherme/disciplina_estatistica_ppgrhs_ufal_aristides/blob/main/PPGRHS_Trab5_estat%C3%ADstica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install lmoments3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 4.7 MB/s eta 0:00:00


In [2]:
#Para utilizar TimesNewRoman
import matplotlib as mpl
import matplotlib.font_manager as fm
print(mpl.__version__)

!wget -O TimesNewRoman.ttf https://github.com/justrajdeep/fonts/raw/master/Times%20New%20Roman.ttf
font_dirs = ["/content/"]
font_files = fm.findSystemFonts(fontpaths=font_dirs, fontext='ttf')
for font_file in font_files:
  print(font_file) if 'TimesNewRoman' in font_file else None
  fm.fontManager.addfont(font_file)

import matplotlib.pyplot as plt
plt.rcParams['font.serif'] = "Times New Roman"
plt.rcParams['font.family'] = "serif"

3.10.0
--2026-05-30 22:32:59--  https://github.com/justrajdeep/fonts/raw/master/Times%20New%20Roman.ttf
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/justrajdeep/fonts/master/Times%20New%20Roman.ttf [following]
--2026-05-30 22:33:00--  https://raw.githubusercontent.com/justrajdeep/fonts/master/Times%20New%20Roman.ttf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 834452 (815K) [application/octet-stream]
Saving to: ‘TimesNewRoman.ttf’

TimesNewRoman.ttf   100%[===================>] 814.89K  --.-KB/s    in 0.01s   

2026-05-30 22:33:00 (82.2 MB/s) - ‘TimesNewRoman.ttf’ saved [834452/834452]

/c

In [28]:
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import lmoments3 as lmom
from lmoments3 import distr
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


In [4]:
df_chuva_max = pd.read_excel('/content/drive/MyDrive/Meus Arquivos /PPGRHS/PPGRHS_Estudos_disciplinas/Estatística/Trabalhos/Trabalho_5/Dados_Trabalho_5.xlsx')

#vamos escolher os dados para a equipe 1
df_chuva_max = df_chuva_max[['Equipe 2']]

In [270]:
fig = px.line(df_chuva_max['Equipe 2'], markers=True)

fig.update_xaxes(title='Vazões Máximas Anuais')
fig.update_yaxes(title='Vazão (m³/s)')


fig.update_layout(title={'text':'Máximas anuais', 'x':0.5}, xaxis_title='Observações', yaxis_title='Vazão (m³/s)',
                  font=dict(family='Times New Roman', size=25, color='black'), plot_bgcolor='white')

fig.update_xaxes(showline=True, mirror=True, linewidth=1, linecolor='black', gridcolor='lightgray')
fig.update_yaxes(showline=True, mirror=True, linewidth=1, linecolor='black', gridcolor='lightgray')

fig.update_traces(line=dict(dash='dash', color='gray', width=1), marker=dict(size=9, color='blue', symbol='circle'))

fig.show()

In [238]:
fig = px.histogram(df_chuva_max['Equipe 2'], nbins=6, histnorm='probability density')
fig.update_layout(bargap=0.05)
fig.update_xaxes(title='Vazão (m³/s)')
fig.update_yaxes(title='Frequência')


fig.update_layout(title={'text':'Histograma - máximas anuais', 'x':0.5}, xaxis_title='Vazão (m³/s)', yaxis_title='Probabilidade relativa (%)',
                  font=dict(family='Times New Roman', size=25, color='black'), plot_bgcolor='white')

fig.update_xaxes(showline=True, mirror=True, linewidth=1, linecolor='black', gridcolor='lightgray')
fig.update_yaxes(showline=True, mirror=True, linewidth=1, linecolor='black', gridcolor='lightgray')


fig.show()

## Ajustando dados às distribuições Gev, Gumbel e Lognormal, pelos métodos:
### Máxima Verossimilhança, Momentos e Momentos-L

### Calculando CDF e tr empíricos

In [21]:
def empirical_dist(data, name_var='prec', method_calc='grigorten'):

    '''  Gera distribuição empírica (CDF) e tempo de retorno
    - data: lista com os dados
    - method_calca: indicar opção de cálculo da prob. de excedência.
                  Digitar: 'weibull', 'gringorten', 'blom', 'hazen', 'cunnane'

                  '''
    #Crianddo dataframe, atribuindo valores dso dados a coluna 'prec'
    df_empirical = pd.DataFrame(columns=[name_var, 'prob_cdf', 'prob_pdf', 'tr'])
    df_empirical[name_var] = data

    #Organizando em ordem crescente e redefinindo o índice
    df_empirical.sort_values(by=name_var, ascending=True, inplace=True)
    df_empirical.reset_index(inplace=True)
    df_empirical.drop(columns=['index'], inplace=True)

    #Selecionando método de cálculo
    dic_method = {'weibull': 0, 'grigorten':0.44, 'blom':0.375, 'hazen':0.5, 'cunanne':0.4}

    #Calculando e armazenando CDF e tempo de retorno
    n = len(df_empirical)

    #Selecionando parâmetro de cálculo do tempo de retorno empírico
    a = dic_method[method_calc]

    #Calculando
    df_empirical['prob_cdf'] = [(i - a) / ((n + 1) - 2*a)  for i in range(1, n + 1)]
    df_empirical['tr'] = 1 / (1 - df_empirical['prob_cdf'])

    return df_empirical


#calculando distribuição empírica
df_empirical = empirical_dist(df_chuva_max['Equipe 2'].to_list(), name_var='vazao', method_calc='grigorten')

### GEV

In [72]:
data = df_chuva_max['Equipe 2'].to_list()

#Ajustando para Gev
fit_gev_mvs = scipy.stats.genextreme.fit(data, method='MLE')
fit_gev_mom = scipy.stats.genextreme.fit(data, method='MM')
fit_gev_lmom = distr.gev.lmom_fit(data)

#visualizando resultados do fit
print(fit_gev_mvs)
print(fit_gev_mom)
print(fit_gev_lmom)

#visualizando
values_vazao_gev = np.linspace(0.95*np.min(data), np.max(data)*1.05, 1000)
cdf_gev_mvs = scipy.stats.genextreme.cdf(values_vazao_gev, *fit_gev_mvs)
cdf_gev_mom = scipy.stats.genextreme.cdf(values_vazao_gev, *fit_gev_mom)
cdf_gev_lmom = scipy.stats.genextreme.cdf(values_vazao_gev, *(fit_gev_lmom['c'], fit_gev_lmom['loc'], fit_gev_lmom['scale']))

(np.float64(0.5237466307432204), np.float64(18.332821802285693), np.float64(2.841769783636366))
(np.float64(0.5057883865182964), np.float64(18.305671436263143), np.float64(2.822225398646615))
OrderedDict({'c': np.float64(0.5472926962481083), 'loc': np.float64(18.35075816991572), 'scale': np.float64(2.893749267221179)})


In [73]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=values_vazao_gev, y=cdf_gev_mvs, name='MVS', line=dict(
              color='red', width=3)))
fig.add_trace(go.Scatter(x=values_vazao_gev, y=cdf_gev_mom, name='MOM', line=dict(
              color='green', width=3)))
fig.add_trace(go.Scatter(x=values_vazao_gev, y=cdf_gev_lmom, name='L-MOM', line=dict(
              color='blue', width=3)))

fig.add_trace(go.Scatter(x=df_empirical['vazao'], y=df_empirical['prob_cdf'], name='Empírico', mode='markers', line=dict(color='black')))
fig.update_layout(title={'text':'GEV', 'x':0.5}, xaxis_title='Vazão (m³/s)', yaxis_title='Prob. acumulada',
                  font=dict(family='Times New Roman', size=25, color='black'), plot_bgcolor='white')

fig.update_xaxes(showline=True, mirror=True, linewidth=1, linecolor='black', gridcolor='lightgray')
fig.update_yaxes(showline=True, mirror=True, linewidth=1, linecolor='black', gridcolor='lightgray')


fig.show()

In [71]:
tr_theorical_gev_mvs = 1/(1-np.array(cdf_gev_mvs))
tr_theorical_gev_mom = 1/(1-np.array(cdf_gev_mom))
tr_theorical_gev_lmom = 1/(1-np.array(cdf_gev_lmom))


fig = go.Figure()
fig.add_trace(go.Scatter(x=tr_theorical_gev_mvs, y=values_vazao_gev, name='MVS', line=dict(
              color='red', width=3)))
fig.add_trace(go.Scatter(x=tr_theorical_gev_mom, y=values_vazao_gev, name='MOM', line=dict(
              color='green', width=3)))
fig.add_trace(go.Scatter(x=tr_theorical_gev_lmom, y=values_vazao_gev, name='L-MOM', line=dict(
              color='blue', width=3)))
fig.add_trace(go.Scatter(x=df_empirical['tr'], y=df_empirical['vazao'], name='Empírico', mode='markers', line=dict(color='black')))
fig.update_layout(title={'text':'GEV', 'x':0.5}, xaxis_title='TR (anos)', yaxis_title='Vazão (m³/s)',
                  font=dict(family='Times New Roman', size=25, color='black'), plot_bgcolor='white')
fig.update_xaxes(showline=True, mirror=True, linewidth=1, linecolor='black', gridcolor='lightgray', range=[0, 1.1*np.max(data)])
fig.update_yaxes(showline=True, mirror=True, linewidth=1, linecolor='black', gridcolor='lightgray')
fig.show()

/tmp/ipykernel_1171/2179708673.py:1: RuntimeWarning:

divide by zero encountered in divide

/tmp/ipykernel_1171/2179708673.py:2: RuntimeWarning:

divide by zero encountered in divide

/tmp/ipykernel_1171/2179708673.py:3: RuntimeWarning:

divide by zero encountered in divide



### Gumbel

In [208]:
#Ajustando Gumbel
fit_gumbel_mvs = scipy.stats.gumbel_r.fit(data, method='MLE')
fit_gumbel_mom = scipy.stats.gumbel_r.fit(data, method='MM')
fit_gumbel_lmom = distr.gum.lmom_fit(data)

#visualizando resultados do fit
print(fit_gumbel_mvs)
print(fit_gumbel_mom)
print(fit_gumbel_lmom)

#visualizando
values_vazao_gumbel = np.linspace(0.95*np.min(data), np.max(data)*1.2, 1000)
cdf_gumbel_mvs = scipy.stats.gumbel_r.cdf(values_vazao_gumbel, *fit_gumbel_mvs)
cdf_gumbel_mom = scipy.stats.gumbel_r.cdf(values_vazao_gumbel, *fit_gumbel_mom)
cdf_gumbel_lmom = scipy.stats.gumbel_r.cdf(values_vazao_gumbel, *(fit_gumbel_lmom['loc'], fit_gumbel_lmom['scale']))

(np.float64(17.558063493735023), 2.8636616321291544)
(np.float64(17.763665245381773), np.float64(2.0368813390212903))
OrderedDict({'loc': np.float64(17.704102079098586), 'scale': np.float64(2.1400718272651456)})


In [271]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=values_vazao_gumbel , y=cdf_gumbel_mvs, name='MVS', line=dict(
              color='red', width=3)))
fig.add_trace(go.Scatter(x=values_vazao_gumbel , y=cdf_gumbel_mom , name='MOM', line=dict(
              color='green', width=3)))
fig.add_trace(go.Scatter(x=values_vazao_gumbel , y=cdf_gumbel_lmom, name='L-MOM', line=dict(
              color='blue', width=3)))
fig.add_trace(go.Scatter(x=df_empirical['vazao'], y=df_empirical['prob_cdf'], name='Empírico', mode='markers', line=dict(color='black')))
fig.update_layout(title={'text':'Gumbel', 'x':0.5}, xaxis_title='Vazão (m³/s)', yaxis_title='Prob. acumulada',
                  font=dict(family='Times New Roman', size=25, color='black'), plot_bgcolor='white')

fig.update_xaxes(showline=True, mirror=True, linewidth=1, linecolor='black', gridcolor='lightgray')
fig.update_yaxes(showline=True, mirror=True, linewidth=1, linecolor='black', gridcolor='lightgray')

fig.show(renderer="notebook_connected")


In [210]:
tr_theorical_gumbel_mvs = 1/(1-np.array(cdf_gumbel_mvs))
tr_theorical_gumbel_mom = 1/(1-np.array(cdf_gumbel_mom))
tr_theorical_gumbel_lmom = 1/(1-np.array(cdf_gumbel_lmom))


fig = go.Figure()
fig.add_trace(go.Scatter(x=tr_theorical_gumbel_mvs, y=values_vazao_gumbel, name='MVS', line=dict(
              color='red', width=3)))
fig.add_trace(go.Scatter(x=tr_theorical_gumbel_mom, y=values_vazao_gumbel, name='MOM', line=dict(
              color='green', width=3)))
fig.add_trace(go.Scatter(x=tr_theorical_gumbel_lmom, y=values_vazao_gumbel, name='L-MOM', line=dict(
              color='blue', width=3)))
fig.add_trace(go.Scatter(x=df_empirical['tr'], y=df_empirical['vazao'], name='Empírico', mode='markers', line=dict(color='black')))
fig.update_layout(title={'text':'Gumbel', 'x':0.5}, xaxis_title='TR (anos)', yaxis_title='Vazão (m³/s)',
                  font=dict(family='Times New Roman', size=25, color='black'), plot_bgcolor='white')
fig.update_xaxes(showline=True, mirror=True, linewidth=1, linecolor='black', gridcolor='lightgray', range=[0, 30])
fig.update_yaxes(showline=True, mirror=True, linewidth=1, linecolor='black', gridcolor='lightgray')
fig.show()

## LogNormal

In [112]:
scipy.stats.lmoment(data)

array([18.93938506,  1.48338475, -0.13724949,  0.14253055])

In [135]:
#Ajustando Gumbel
fit_lognormal_mvs = scipy.stats.lognorm.fit(data, method='MLE')
fit_lognormal_mom = scipy.stats.lognorm.fit(data, method='MM')

#visualizando resultados do fit
print(fit_lognormal_mvs)
print(fit_lognormal_mom)

#visualizando
values_vazao_lognormal = np.linspace(0.95*np.min(data), np.max(data)*1.05, 1000)
cdf_lognorm_mvs = scipy.stats.lognorm.cdf(values_vazao_lognormal, *fit_lognormal_mvs)
cdf_lognorm_mom = scipy.stats.lognorm.cdf(values_vazao_lognormal, *fit_lognormal_mom)


(np.float64(9.965314149766822e-06), -262131.37451537643, np.float64(262150.3138874215))
(np.float64(0.506867889778537), np.float64(14.475095772085474), np.float64(3.9403039391646413))


In [137]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=values_vazao_lognormal, y=cdf_lognorm_mvs, name='MVS', line=dict(
              color='red', width=3)))
fig.add_trace(go.Scatter(x=values_vazao_lognormal, y=cdf_lognorm_mom, name='MOM', line=dict(
              color='green', width=3)))

fig.add_trace(go.Scatter(x=df_empirical['vazao'], y=df_empirical['prob_cdf'], name='Empírico', mode='markers', line=dict(color='black')))
fig.update_layout(title={'text':'LogNormal', 'x':0.5}, xaxis_title='Vazão (m³/s)', yaxis_title='Prob. acumulada',
                  font=dict(family='Times New Roman', size=25, color='black'), plot_bgcolor='white')

fig.update_xaxes(showline=True, mirror=True, linewidth=1, linecolor='black', gridcolor='lightgray')
fig.update_yaxes(showline=True, mirror=True, linewidth=1, linecolor='black', gridcolor='lightgray')


fig.show()

In [134]:
lmom.fstats.lnorm(data)

AttributeError: module 'lmoments3' has no attribute 'fstats'

## Weibull

In [170]:
#Ajustando Gumbel
fit_weibull_mvs = scipy.stats.weibull_max.fit(data, method='MLE')
fit_weibull_mom = scipy.stats.weibull_max.fit(data, method='MM')
fit_weibull_lmom = distr.wei.lmom_fit(data)

#visualizando resultados do fit
print(fit_weibull_mvs)
print(fit_weibull_mom)
print(fit_weibull_lmom)

#visualizando
values_vazao_weibull = np.linspace(0.95*np.min(data), np.max(data)*1.05, 1000)
cdf_weibull_mvs = scipy.stats.weibull_max.cdf(values_vazao_lognormal, *(fit_weibull_mvs[0], fit_weibull_mvs[1], fit_weibull_mvs[2]))
cdf_weibull_mom = scipy.stats.weibull_max.cdf(values_vazao_lognormal, *(fit_weibull_mom[0], fit_weibull_mom[1], fit_weibull_mom[2]))
cdf_weibull_lmom = scipy.stats.weibull_min.cdf(values_vazao_lognormal, *(fit_weibull_lmom['c'], fit_weibull_lmom['loc'], fit_weibull_lmom['scale']))


(np.float64(0.6008804667286896), np.float64(23.299927200734576), np.float64(1.5189794545151067))
(np.float64(1.962148337647796), np.float64(23.853620144801923), np.float64(5.54303942539358))
OrderedDict({'c': np.float64(19.4155692797043), 'loc': np.float64(-23.357433188840318), 'scale': np.float64(43.48049678314028)})


In [171]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=values_vazao_weibull, y=cdf_weibull_mvs, name='MVS', line=dict(
              color='red', width=3)))
fig.add_trace(go.Scatter(x=values_vazao_weibull, y=cdf_weibull_mom, name='MOM', line=dict(
              color='green', width=3)))
fig.add_trace(go.Scatter(x=values_vazao_weibull, y=cdf_weibull_lmom, name='L-MOM', line=dict(
              color='blue', width=3)))
fig.add_trace(go.Scatter(x=df_empirical['vazao'], y=df_empirical['prob_cdf'], name='Empírico', mode='markers', line=dict(color='black')))

fig.update_layout(title={'text':'Weibull', 'x':0.5}, xaxis_title='Vazão (m³/s)', yaxis_title='Prob. acumulada',
                  font=dict(family='Times New Roman', size=25, color='black'), plot_bgcolor='white')

fig.update_xaxes(showline=True, mirror=True, linewidth=1, linecolor='black', gridcolor='lightgray')
fig.update_yaxes(showline=True, mirror=True, linewidth=1, linecolor='black', gridcolor='lightgray')

fig.show()

In [186]:
tr_theorical_weibull_mvs = 1/(1-np.array(cdf_weibull_mvs))
tr_theorical_weibull_mom = 1/(1-np.array(cdf_weibull_mom))
tr_theorical_weibull_lmom = 1/(1-np.array(cdf_weibull_lmom))


fig = go.Figure()
fig.add_trace(go.Scatter(x=tr_theorical_weibull_mvs, y=values_vazao_weibull, name='MVS', line=dict(
              color='red', width=3)))
fig.add_trace(go.Scatter(x=tr_theorical_weibull_mom, y=values_vazao_weibull, name='MOM', line=dict(
              color='green', width=3)))
fig.add_trace(go.Scatter(x=tr_theorical_weibull_lmom, y=values_vazao_weibull, name='L-MOM', line=dict(
              color='blue', width=3)))
fig.add_trace(go.Scatter(x=df_empirical['tr'], y=df_empirical['vazao'], name='Empírico', mode='markers', line=dict(color='black')))
fig.update_layout(title={'text':'weibull', 'x':0.5}, xaxis_title='TR (anos)', yaxis_title='Vazão (m³/s)',
                  font=dict(family='Times New Roman', size=25, color='black'), plot_bgcolor='white')
fig.update_xaxes(showline=True, mirror=True, linewidth=1, linecolor='black', gridcolor='lightgray', range=[0, 30])
fig.update_yaxes(showline=True, mirror=True, linewidth=1, linecolor='black', gridcolor='lightgray')
fig.show()

/tmp/ipykernel_1171/968739380.py:1: RuntimeWarning:

divide by zero encountered in divide

/tmp/ipykernel_1171/968739380.py:2: RuntimeWarning:

divide by zero encountered in divide



## Comparando as três distribuições via momentos-L

In [211]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=values_vazao_gev, y=cdf_gev_lmom, name='Gev', line=dict(
              color='red', width=3)))
fig.add_trace(go.Scatter(x=values_vazao_gumbel, y=cdf_gumbel_lmom, name='Gumbel', line=dict(
              color='green', width=3)))
fig.add_trace(go.Scatter(x=values_vazao_weibull, y=cdf_weibull_lmom, name='Weibull', line=dict(
              color='blue', width=3)))
fig.add_trace(go.Scatter(x=df_empirical['vazao'], y=df_empirical['prob_cdf'], name='Empírico', mode='markers', line=dict(color='black')))

fig.update_layout(title={'text':'Momentos-L', 'x':0.5}, xaxis_title='Vazão (m³/s)', yaxis_title='Prob. acumulada',
                  font=dict(family='Times New Roman', size=25, color='black'), plot_bgcolor='white')

fig.update_xaxes(showline=True, mirror=True, linewidth=1, linecolor='black', gridcolor='lightgray')
fig.update_yaxes(showline=True, mirror=True, linewidth=1, linecolor='black', gridcolor='lightgray')

fig.show()

In [212]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=tr_theorical_gev_lmom, y=values_vazao_gev, name='Gev', line=dict(
              color='red', width=3)))
fig.add_trace(go.Scatter(x=tr_theorical_gumbel_lmom, y=values_vazao_gumbel, name='Gumbel', line=dict(
              color='green', width=3)))
fig.add_trace(go.Scatter(x=tr_theorical_weibull_lmom, y=values_vazao_weibull, name='Weibull', line=dict(
              color='blue', width=3)))
fig.add_trace(go.Scatter(x=df_empirical['tr'], y=df_empirical['vazao'], name='Empírico', mode='markers', line=dict(color='black')))
fig.update_layout(title={'text':'Momentos-L', 'x':0.5}, xaxis_title='TR (anos)', yaxis_title='Vazão (m³/s)',
                  font=dict(family='Times New Roman', size=25, color='black'), plot_bgcolor='white')
fig.update_xaxes(showline=True, mirror=True, linewidth=1, linecolor='black', gridcolor='lightgray', range=[0, 50])
fig.update_yaxes(showline=True, mirror=True, linewidth=1, linecolor='black', gridcolor='lightgray')
fig.show()

### Calculando vazão para TR = 25 anos

In [213]:
#para gev
df_trs_gev = pd.DataFrame()
df_trs_gev['vazao'] = values_vazao_gev
df_trs_gev['tr'] = tr_theorical_gev_lmom

#para gumbel
df_trs_gumbel = pd.DataFrame()
df_trs_gumbel['vazao'] = values_vazao_gumbel
df_trs_gumbel['tr'] = tr_theorical_gumbel_lmom

#para weibull
df_trs_weibull = pd.DataFrame()
df_trs_weibull['vazao'] = values_vazao_weibull
df_trs_weibull['tr'] = tr_theorical_weibull_lmom

In [214]:
print('Gev: ', df_trs_gev.loc[df_trs_gev['tr'] >= 25].head(1))
print('Gumbel: ', df_trs_gumbel.loc[df_trs_gumbel['tr'] >= 25].head(1))
print('Weibull: ', df_trs_weibull.loc[df_trs_weibull['tr'] >= 25].head(1))

Gev:           vazao         tr
852  22.629894  25.144736
Gumbel:           vazao        tr
786  24.555814  25.07587
Weibull:           vazao         tr
868  22.829625  25.277093
